<a href="https://colab.research.google.com/github/RenatGreen-flag/Model-Liniar-Regression/blob/main/linearRegressionGoldnIDR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**INFORMASI DATASET**

Dataset ini berisi data historis **Harga Emas Dunia dan Indeks USD/IDR** yang dikumpulkan oleh Yahoo Finance dalam rentang waktu **2023 hingga sekarang**.

**Sumber Data**: Yahoo Finance - Gold Price, Indeks USD/IDR (2023-Sekarang)

**Parameter (Fitur)**:

* usdidr_close: Indeks USD ke IDR Tutup

**Target**: gold_cLose (Harga Komoditas Emas Dunia Tutup)








In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
!pip install ydata-profiling
from ydata_profiling import ProfileReport
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

**# Langkah 1 Mengambil Data Dari Library YFinance**



1.   Ambil data harga emas dan indeks nilsi tukar USD/IDR
2.   Gunakan ticker symbol yang tersedia di yfinance.
3. Gunakan dictionary untuk memudahkan pengambilan data.
4. Tentukan rentang pengambilan data, saya menggunakan data historis dari januari 2023 - sekarang.
5.




In [4]:
TROY_OUNCE_TO_GRAM = 31.1035   # 1 troy ounce = 31.1035 gram
pd.set_option('display.float_format', lambda x: '%.2f' % x)
ticker_map = {
    "GC=F"    : "gold",
    "USDIDR=X": "usdidr",
}

START = "2023-01-01"
END   = datetime.today().strftime("%Y-%m-%d")

LAG_STEPS       = [1, 3, 7]
ROLLING_WINDOWS = [3, 7]

In [5]:
raw = yf.download(
    tickers     = list(ticker_map.keys()),
    start       = START,
    end         = END,
    auto_adjust = True,
    progress    = True
)

df = raw["Close"].rename(columns=ticker_map).add_suffix("_close")
print(f"\nShape setelah download: {df.shape}")
print(df)
print(f"Rentang tanggal awal  : {df.index.min().date()} s.d. {df.index.max().date()}")

[*********************100%***********************]  2 of 2 completed


Shape setelah download: (905, 2)
Ticker      gold_close  usdidr_close
Date                                
2023-01-02         NaN      15508.80
2023-01-03     1839.70      15550.00
2023-01-04     1852.80      15571.80
2023-01-05     1834.80      15560.00
2023-01-06     1864.20      15620.00
...                ...           ...
2026-06-19         NaN      17794.40
2026-06-22     4181.90      17788.00
2026-06-23     4129.90      17862.00
2026-06-24     3990.30      17916.00
2026-06-25     4030.50      17990.00

[905 rows x 2 columns]
Rentang tanggal awal  : 2023-01-02 s.d. 2026-06-25


In [6]:
# TAHAP 2: PENYELARASAN TIMEZONE (SHIFT +1 HARI)
GLOBAL_TICKERS = list(ticker_map.keys())

for ticker_name in GLOBAL_TICKERS:
  col = f"{ticker_name}_close"
  if col in df.columns:
    df[col] = df[col].shift(1)

In [7]:
# TAHAP 3: REINDEX KE KALENDER HARIAN PENUH

full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq="D")
df = df.reindex(full_range)
df.index.name = "Date"

print(f"Shape telah reindex: {df.shape}")
print(f"Rentang tanggal baru: {df.index.min().date()} s.d. {df.index.max().date()}")

Shape telah reindex: (1271, 2)
Rentang tanggal baru: 2023-01-02 s.d. 2026-06-25


In [8]:
# TAHAP 4: FORWARD FILL (FFILL) -> HARGA CLOSE HARI TERAKHIR

price_cols = [c for c in df.columns if c.endswith("_close")]
df[price_cols] = df[price_cols].ffill().bfill()

missing_after = df[price_cols].isnull().sum()
print(f"\nJumlah missing value setelah forward fill:\n{missing_after}")
print()



Jumlah missing value setelah forward fill:
Ticker
gold_close      0
usdidr_close    0
dtype: int64



In [9]:
# # feature engineering gold usd -> idr per gram
df["gold_close_idr_per_oz"] = df["gold_close"] * df['usdidr_close']

df["gold_close_idr_per_gram"] = df["gold_close_idr_per_oz"] / TROY_OUNCE_TO_GRAM

price_cols = price_cols + ["gold_close_idr_per_oz","gold_close_idr_per_gram"]
print(df[["gold_close", "usdidr_close", "gold_close_idr_per_oz", "gold_close_idr_per_gram"]].tail(3).to_string())

Ticker      gold_close  usdidr_close  gold_close_idr_per_oz  gold_close_idr_per_gram
Date                                                                                
2026-06-23     4129.90      17862.00            73768272.06               2371703.25
2026-06-24     3990.30      17916.00            71490215.67               2298462.09
2026-06-25     4030.50      17990.00            72508695.00               2331206.94


In [10]:
# 2. Suruh YData buat "cetak biru" analisisnya
profile = ProfileReport(df, title="Laporan Analisis Data Melzzz")

# 3. Simpan hasilnya jadi halaman web (HTML)!
profile.to_file("laporan_web_kamu.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 4/4 [00:00<00:00, 58.19it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
dataclean = df[["gold_close", "usdidr_close", "gold_close_idr_per_oz", "gold_close_idr_per_gram"]]
#Setelah mendapatkan dataclean . hapus field antara X dan Y

df['harga_besok_per_gram'] = df['gold_close_idr_per_gram'].shift(-1)
df = df.dropna()
dataclean = df[["gold_close", "usdidr_close", "gold_close_idr_per_oz", "gold_close_idr_per_gram", "harga_besok_per_gram"]]
dataclean.tail()

x = dataclean[["gold_close", "usdidr_close", "gold_close_idr_per_oz", "gold_close_idr_per_gram"]]
y = dataclean["harga_besok_per_gram"]

In [12]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, shuffle=False)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
score = model.score(X_test, y_test)
print(f"R2 Score: {score} ")
print(f"R2 Score: {r2_score(y_test, y_pred)}")
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")

R2 Score: 0.9534103085355896 
R2 Score: 0.9534103085355896
Mean Squared Error: 1874556476.6139057
